# Projeto #4 - Planejamento de Capacidade na Nuvem

## Equipe

- Carlos Duarte - matr. 2527530
- Jonas de A. Luz Jr. - matr. 2519171

----

In [1]:
import os, re

import pandas as pd

## Objetivo
>
> Fonte: [Especificação do projeto 4](https://docs.google.com/document/d/13QL64Om-XBFfEqDDyZp8czq-vdWQG-e4mvE3uwRpi-c/edit?tab=t.0)

**Da expecificação original:**

- Arquitetar a infraestrutura de um serviço de blog (WordPress) na AWS.
- Desafio: Maximizar o RPS (Requests Per Second) suportado pelo serviço, sujeito às seguintes restrições:
  - Orçamento: O custo da sua camada de aplicação não pode exceder US$0.50/hora (preço On-Demand us-east-1).
  - Qualidade (SLO): Taxa de Erro < 1% e Latência P95 < 10000ms.
  - Componentes Fixos: O Banco de Dados e o Load Balancer são fornecidos pela "Arena" e não podem ser modificados.
- O trabalho é ajustar a camada de aplicação, escolhendo a melhor combinação de escalabilidade vertical (tamanho da máquina) e horizontal (quantidade de máquinas) para implantar o WordPress.

São fornecidos os *scripts base* para a implementação do WordPress, que podem ser encontrados na pasta `scripts` do projeto.

## Estratégia de Implementação

Nossa estratégia de implementação do trabalho foi a seguinte:

1. Converter os scripts originais para o formato de *batch* do PowerShell, uma vez que o trabalho foi realizado em ambiente de desenvolvimento Windows.
2. Levantar os custos operacionais das instâncias de teste, uma vez que o orçamento foi limitado a US$0.50/hora. Os custos oficiais da AWS foram consultados [no site oficial do serviço EC2](https://us-east-1.console.aws.amazon.com/ec2/home?region=us-east-1).
3. Definição de experimentos para teste de escalabilidade vertical e horizontal, com o objetivo de encontrar a melhor combinação de tamanho da máquina e quantidade de máquinas para implantar o WordPress.
4. Implementação de scripts de apoio com o objetivo de automatizar a execução dos experimentos e coleta de métricas.
5. Implementação de scripts de apoio para coleta de métricas e análise de resultados.
6. Realização de modificações na configuração da aplicação WordPress para otimização do desempenho.
7. Avaliação dos resultados dos experimentos e coleta de métricas.
8. Elaboração de relatório com os resultados dos experimentos e coleta de métricas e escolha da melhor combinação de tamanho da máquina e quantidade de máquinas para implantar o WordPress.

O detalhamento das etapas é descrito nas seções seguintes.

## Implementação

### Conversão dos Scripts para o PowerShell

Os scripts originais foram convertidos para o formato de *batch* do PowerShell, uma vez que o trabalho foi realizado em ambiente de desenvolvimento Windows.

Os novos scripts constam na pasta `scripts` do projeto, enquanto os scripts originais foram preservados na subpasta `scripts/_original_bash_scripts`.

Foram mantidos em formato bash os scripts que, na verdade, são transferidos para as instâncias de teste, guardados na subpasta `scripts/data_scripts`.

### Levantamento de Custos Operacionais

Os custos operacionais das instâncias de teste foram obtidos [no site oficial do serviço EC2](https://us-east-1.console.aws.amazon.com/ec2/home?region=us-east-1). Os dados extraídos da página de preços do EC2 foram consolidados em um DataFrame Pandas.

In [2]:
# Dados de tipos de instância baixados do site de preços oficial do EC2, com custo menos que US$0.50/hora
#
PRICES_DATA_PATH = "data/ec2-prices"
data_files = os.listdir(PRICES_DATA_PATH)
print(f"Encontrados {len(data_files)} arquivos. Iniciando extração de dados...")

df_prices = pd.DataFrame()
for csv_file in data_files:
    print(csv_file, end='... ')
    
    csv_partial_file = os.path.join(PRICES_DATA_PATH, csv_file)
    df_prices = pd.concat([df_prices, pd.read_csv(csv_partial_file)], ignore_index=True)

print(f"\nRegistradas {df_prices.shape[0]} linhas.")

Encontrados 7 arquivos. Iniciando extração de dados...
instancetypes-p1.csv... instancetypes-p2.csv... instancetypes-p3.csv... instancetypes-p4.csv... instancetypes-p5.csv... instancetypes-p6.csv... instancetypes-p7.csv... 
Registradas 339 linhas.


In [3]:
# Convertendo valor do custo e selecionando os tipos de instância candidatos.
#
PRICE_COLUMN = 'On-Demand Linux pricing'

float_extractor = lambda v: float(re.findall(r"[-+]?\d*\.\d+|\d+", str(v))[0])

df_prices['Cost'] = df_prices[PRICE_COLUMN].apply(float_extractor)
df_prices = df_prices[(df_prices['Cost'] > 0) & (df_prices['Cost'] <= 0.5)]

print(f"Filtradas {df_prices.shape[0]} linhas com custo menor que US$0.50/hora.")

Filtradas 327 linhas com custo menor que US$0.50/hora.


Com a lista de tipos de instância candidatos inicial, foram excluídas as famílias de tipos que não interessam ou não se aplicam ao problema, com base nas [descrições oficiais de cada tipo de instância](https://docs.aws.amazon.com/ec2/latest/instancetypes/instance-type-names.html), como as famílias iniciadas por `g`, voltadas para aceleração por GPU ou `h` e `d`, que utilizam armazenamento HDD, dentre outras.

Por esta regra, foram selecionados os tipos de instância `t3`, por ter sido utilizado como exemplo do problema, instâncias das famílias `c`, otimizadas para computação e algumas outras da família `r`, otimizadas para memória. Foram também removidas as famílias variantes, como, por exemplo, aquelas que especificam o processador -- `c5a`, que indica uso de AMD, ou `c6i` que especifica o uso de Intel, etc -- ou variantes que modificam o armazenamento ou rede. Do restante, selecionamos apenas a versão mais recente do tipo. Assim, restaram para análise os tipos de instância `t3`, `c5`, `m5` e `r5`.

In [4]:
INSTANCE_TYPE_COLUMN = 'Instance type'

families = sorted(
    df_prices[INSTANCE_TYPE_COLUMN].str.split('.').str[0].unique()
)

print(f"Famílias existentes: {families}")

Famílias existentes: ['a1', 'c1', 'c3', 'c4', 'c5', 'c5a', 'c5ad', 'c5d', 'c5n', 'c6a', 'c6g', 'c6gd', 'c6gn', 'c6i', 'c6id', 'c6in', 'c7a', 'c7g', 'c7gd', 'c7gn', 'c7i', 'c7i-flex', 'c8a', 'c8g', 'c8gb', 'c8gd', 'c8gn', 'c8i', 'c8i-flex', 'd3', 'g4ad', 'g5g', 'g6f', 'h1', 'i3', 'i3en', 'i4g', 'i4i', 'i7i', 'i7ie', 'i8g', 'i8ge', 'im4gn', 'inf1', 'is4gen', 'm1', 'm2', 'm3', 'm4', 'm5', 'm5a', 'm5ad', 'm5d', 'm5dn', 'm5n', 'm5zn', 'm6a', 'm6g', 'm6gd', 'm6i', 'm6id', 'm6idn', 'm6in', 'm7a', 'm7g', 'm7gd', 'm7i', 'm7i-flex', 'm8a', 'm8g', 'm8gb', 'm8gd', 'm8gn', 'm8i', 'm8i-flex', 'r3', 'r4', 'r5', 'r5a', 'r5ad', 'r5b', 'r5d', 'r5dn', 'r5n', 'r6a', 'r6g', 'r6gd', 'r6i', 'r6id', 'r6idn', 'r6in', 'r7a', 'r7g', 'r7gd', 'r7i', 'r7iz', 'r8a', 'r8g', 'r8gb', 'r8gd', 'r8gn', 'r8i', 'r8i-flex', 't1', 't2', 't3', 't3a', 't4g', 'x2gd', 'x8g', 'z1d']


In [5]:
filter = lambda x: x.startswith('t3.') or x.split('.')[0] in ['c5', 'm5', 'r5', 't3']

df_prices = df_prices[df_prices[INSTANCE_TYPE_COLUMN].apply(filter)]

print(f"Filtradas {df_prices.shape[0]} linhas com famílias de instância de interesse.")

Filtradas 14 linhas com famílias de instância de interesse.


Para cada uma das famílias candidatas, calculamos a quantidade máxima de instâncias que podem ser implantadas no orçamento de US$0.50/hora.

In [6]:
max_calculator = lambda x: int(0.5 / x)

df_prices.loc[:, 'Max Instances'] = df_prices['Cost'].apply(max_calculator)

In [7]:
df_prices [['Instance type', 'Cost', 'Max Instances']].sort_values(by='Max Instances', ascending=False)

,Instance type,Cost,Max Instances
2,t3.micro,0.0104,48
7,t3.small,0.0208,24
18,t3.medium,0.0416,12
57,t3.large,0.0832,6
61,c5.large,0.0850,5
79,m5.large,0.0960,5
111,r5.large,0.1260,3
148,t3.xlarge,0.1664,3
152,c5.xlarge,0.1700,2
176,m5.xlarge,0.1920,2


A partir da tabela de preços, foram identificadas os tipos de instância candidatos para o experimento, tendo sido selecionadas as instâncias t3.micro (tipo base, mais barato e mais leve, utilizado como exemplo na especificação do trabalho), c5.large, c5.xlarge e c5.2xlarge (tipo premium, mais caro e mais pesado). Estas escolhas visavam permitir os testes de escalabilidade horizontal e vertical, conforme definido na especificação do trabalho. 

### Experimentos

Os experimentos foram padronizados para testar cada configuração com um certo número de usuários a serem simulados com o Locust. As faixas definidas são as seguintes:

| Cenário | Quantidade de Usuários | 
| --- | --- |
| Uso Mínimo | 100 | 
| Uso Baixo | 250 |
| Uso Médio | 500 |
| Uso Alto | 1000 |
| Uso Muito Alto | 2000 |

Salinta-se que nem todas as faixas foram testadas com todos os tipos de instâncias. Na verdade, iniciamos os testes considerando o uso médio e, à medida que verificamos a impossibilidade de manutenção do orçamento exigido, baixamos o nível para 250 usuários e, por fim, padronizamos a parametrizaçção para uso mínimo, de 100 usuários.

Além disso, os experimentos foram realizados com um tempo de duração de 3 minutos.

#### Análise de escalabilidade - Família `t3`

Iniciamos executando a configuração base (*baseline*), que consiste em uma instância t3.micro com 100 usuários.

In [8]:
def evaluate_results(df_locust: pd.DataFrame):
    """
    Avalia os resultados de um teste de carga com locust.
    """
    df_locust['Error Rate'] = df_locust['Failure Count'] / df_locust['Request Count']
    df_locust['Error Rate Pass'] = df_locust['Error Rate'] < 0.01
    df_locust['P95 Pass'] = df_locust['95%'] < 10000
    df_locust['Total Pass'] = df_locust['Error Rate Pass'] & df_locust['P95 Pass']

    return df_locust[['Type', 'Name', 'Request Count', 'Failure Count', 'Error Rate', 'Error Rate Pass', '95%', 'P95 Pass', 'Total Pass']]


def evaluate_results_from_csv(file_path: str):
    df_locust = pd.read_csv(file_path)
    return evaluate_results(df_locust)

In [9]:
df_baseline = pd.read_csv('results/ProjectExampleBaseline_stats.csv')

evaluate_results(df_baseline)

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,610,598,0.980328,False,8400,True,False
1,GET,/post-detail,1995,1970,0.987469,False,8500,True,False
2,NaN,Aggregated,2605,2568,0.985797,False,8500,True,False


A configuração *baseline*, embora atenda ao requisito mínimo de latência P95, possui taxa de erro de quase 100%, não satisfazendo os requisitos de desempenho. Resolvemos então testar a escalabilidade horizontal da `t3.micro` elevando o número de instâncias para 12 (um quarto do máximo de instâncias disponíveis) e verificando sua capacidade de atender a 100 usuários.

In [10]:
evaluate_results_from_csv('results/t3-micro_100u_12i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1241,85,0.068493,False,3000,True,False
1,GET,/post-detail,4243,301,0.070940,False,3000,True,False
2,NaN,Aggregated,5484,386,0.070387,False,3000,True,False


A taxa de erros com 12 instâncias reduziu-se substancialmnente, assim como a latência P95. Entretanto, a taxa de erros ainda não atingiu o parâmetro exigido de menos de 1%. Vamos aumentar para 24 instâncias, metade do limite máximo de 48 instâncias. 

In [11]:
evaluate_results_from_csv('results/t3-micro_100u_24i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,1658,0,0.0,True,650,True,True
1,GET,/post-detail,5766,0,0.0,True,620,True,True
2,NaN,Aggregated,7424,0,0.0,True,620,True,True


Agora sim. Com 24 instâncias t3.micro, conseguimos atender aos requisitos exigidos pelo experimento. Neste caso, o custo operacional é:

In [12]:
def cost_of(type: str, instances: int) -> str:
    """
    Calcula o custo horário de uma configuração de instâncias EC2.
    """
    ucost = df_prices[df_prices[INSTANCE_TYPE_COLUMN] == type]['Cost'].values[0]
    cost = ucost * instances
    return f"US$ {cost:.2f}"


In [13]:
cost_of('t3.micro', 24)

'US$ 0.25'

Vamos verificar como esta configuração se comporta com 250 usuários.

In [14]:
evaluate_results_from_csv('results/t3-micro_250u_24i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,2646,1125,0.425170,False,4100,True,False
1,GET,/post-detail,9004,3800,0.422035,False,4100,True,False
2,NaN,Aggregated,11650,4925,0.422747,False,4100,True,False


Com 250 usuários, os critérios exigidos não são mais atendidos pela configuração. Vamos extrapolar o uso da `t3.micro` testando-a com 48 instâncias (o máximo possível) para 1000 usuários.

In [15]:
evaluate_results_from_csv('results/t3-micro_1000u_48i_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,9804,7225,0.736944,False,11000,False,False
1,GET,/post-detail,32891,24044,0.731021,False,11000,False,False
2,NaN,Aggregated,42695,31269,0.732381,False,11000,False,False


Novamente, os resultados são muito ruins. Nossos resultados apontam para que o uso da `t3.micro` não é viável para níveis maiores de demanda. 
Vamos seguir testando a família `t3`, buscando identificar um tipo que seja capaz de atender 500 usuários dentro dos critérios de desempenho exigidos.

In [16]:
def review_instances(type_family: str) -> pd.DataFrame:
    """
    Filtra e exibe os tipos de instância da família especificada.
    """
    columns = ['Instance type', 'Instance family', 'Cost', 'Max Instances']
    
    return df_prices[df_prices['Instance family'].eq(type_family)][columns]


In [17]:
review_instances('t3')

,Instance type,Instance family,Cost,Max Instances
2,t3.micro,t3,0.0104,48
7,t3.small,t3,0.0208,24
18,t3.medium,t3,0.0416,12
57,t3.large,t3,0.0832,6
148,t3.xlarge,t3,0.1664,3
253,t3.2xlarge,t3,0.3328,1


Iniciamos os testes utilizando metade da quantidade de instâncias possíveis.

In [72]:
def combine_aggregated_results(csv_files: list[str]) -> pd.DataFrame:
    """
    Combina os resultados de vários arquivos CSV.
    """
    results = pd.DataFrame()

    for file in csv_files:
        df = evaluate_results_from_csv(file)
        file = file.replace('tuned_', '')

        df['Instance type'] = file.split('_')[0].replace('-', '.').split('/')[-1]
        df['Users count'] = int(file.split('_')[1].split('u')[0])
        df['Instance count'] = int(file.split('_')[2].split('i')[0])
        df = df[df['Name'] == 'Aggregated']
        df.drop(columns=['Type', 'Name'], inplace=True)
        df.set_index('Instance type', inplace=True)
        
        results = pd.concat([results, df])

    return results


In [19]:
t3_results_csvs = [
    'results/t3-small_500u_12i_stats.csv',
    'results/t3-medium_500u_6i_stats.csv',
    'results/t3-large_500u_3i_stats.csv',
    'results/t3-xlarge_500u_1i_stats.csv'
]

combine_aggregated_results(t3_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
t3.small,10300,3688,0.358058,False,38000,False,False,500,12
t3.medium,5440,1863,0.342463,False,60000,False,False,500,6
t3.large,6505,4048,0.622291,False,60000,False,False,500,3
t3.xlarge,7914,6238,0.788223,False,10000,False,False,500,1


Algumas observações sobre os resultados:

- Nenhuma das configurações conseguiu atingir os objetivos de performance estabelecidos.
- Embora os tipos de instância sejam progressivamente mais poderosos, a performance não melhora proporcionalmente com a escalabilidade. De fato, metade da quantidade máxima de instâncias `t3-small` têm latência P95 menor que as metades das quantidades máximas de instâncias de `t3-medium` e `t3-large`. No caso desta última, a taxa de erros é quase o dobro que a dos tipos anteriores.
- Embora se aproxime do limite estabelecido para P95, o tipo `t3-xlarge` não atinge a taxa de erros desejada, chegando a quase 80% de erro.

Vamos avaliar novamente estas configurações com o máximo de instâncias possível para cada uma. Aproveitamos para fazer o teste com a `t3.2xlarge`, da qual somente temos disponibilidade de uma instância, respeitando nosso limite de custo.

In [20]:
t3_results_csvs = [
    'results/t3-small_500u_24i_stats.csv',
    'results/t3-medium_500u_12i_stats.csv',
    'results/t3-large_500u_6i_stats.csv',
    'results/t3-xlarge_500u_3i_stats.csv',
    'results/t3-2xlarge_500u_1i_stats.csv'
]

combine_aggregated_results(t3_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
t3.small,11592,4923,0.424689,False,21000,False,False,500,24
t3.medium,10313,3946,0.382624,False,36000,False,False,500,12
t3.large,4828,1051,0.217688,False,60000,False,False,500,6
t3.xlarge,7334,2689,0.366648,False,60000,False,False,500,3
t3.2xlarge,8880,5597,0.630293,False,10000,False,False,500,1


As observações que se pode depreender dos resultados obtidos para 500 usuários são:

- No caso da `t3.small` e `t3.medium`, houve melhoria de P95 ao custo de aumento nas taxas de erro. 
- No caso da `t3.large`, houve redução da taxa de erro sem prejuízo da latência P95.
- No caso da `t3.xlarge`, houve melhoria da taxa de erro com sério comprometimento de P95.
- No caso da `t3.2xlarge`, não há comparação anterior, mas as taxas de erro se apresentam como as maiores dentre o grupo de tipos de instância.
- **Nenhuma das configurações testadas nesta seção atende aos requisitos de latência e taxa de erro.**

**Conclusão:**
- O cenário de 500 usuários não é atendido pela família `t3`.

Vamos baixar o nível para 100 usuários e testar novamente.

In [21]:
t3_results_csvs = [
    'results/t3-micro_100u_48i_stats.csv',
    'results/t3-small_100u_24i_stats.csv',
    'results/t3-medium_100u_12i_stats.csv',
    'results/t3-large_100u_6i_stats.csv',
    'results/t3-xlarge_100u_3i_stats.csv',
    'results/t3-2xlarge_100u_1i_stats.csv'
]

combine_aggregated_results(t3_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48
t3.small,7370,54,0.007327,True,780,True,True,100,24
t3.medium,6542,0,0.000000,True,1900,True,True,100,12
t3.large,3267,35,0.010713,False,6400,True,False,100,6
t3.xlarge,3323,53,0.015949,False,15000,False,False,100,3
t3.2xlarge,3219,33,0.010252,False,38000,False,False,100,1


Observações sobre os resultados com a família `t3`:

- Além do `t3.micro`, também atendem aos critérios de desempenho, os tipos `t3.small`, e `t3.medium`.
- O tipo `t3.large` não atende aos critérios de desempenho por falhar na taxa de erros, passando muito pouco dos 1% exigidos.
- Os tipos `t3.xlarge` e `t3.2xlarge` não atendem aos critérios de desempenho por falhar na taxa de erros e no P95.

Podemos agora, começar a estabelecer os tipos de instância candidatos para nossa solução.

In [70]:
def consolidate_results(selected_csvs: list[str]) -> pd.DataFrame:
    """
    Consolida os resultados combinados, incluindo os custos. 
    """
    df_results = combine_aggregated_results(selected_csvs)

    # Recupera o custo de cada tipo de instância.
    df_results = df_results.merge(df_prices[['Instance type', 'Cost']], on='Instance type', how='left')
    df_results['Total Cost'] = df_results['Cost'] * df_results['Instance count']

    # Consolida indexação do DataFrame.
    df_results.set_index('Instance type', inplace=True)

    df_results.sort_values(by='95%', ascending=True, inplace=True)
    return df_results

In [50]:
t3_candidates = [
    'results/t3-micro_100u_48i_stats.csv',
    'results/t3-micro_100u_24i_stats.csv',
    'results/t3-small_100u_24i_stats.csv',
    'results/t3-medium_100u_12i_stats.csv',
]

consolidate_results(t3_candidates)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48,0.0104,0.4992
t3.micro,7424,0,0.000000,True,620,True,True,100,24,0.0104,0.2496
t3.small,7370,54,0.007327,True,780,True,True,100,24,0.0208,0.4992
t3.medium,6542,0,0.000000,True,1900,True,True,100,12,0.0416,0.4992


Como buscamos o maior desempenho posível, podemos retirar a alternativa de usar 24 instâncias de `t3.micro`, já que, usando o máximo de 48 instâncias deste tipo, conseguimos manter o orçamento com maior desempenho.

Nossa tabela de tipos candidatos, por agora, fica: 

In [51]:
candidate_csvs = t3_candidates = [
    'results/t3-micro_100u_48i_stats.csv',
    'results/t3-small_100u_24i_stats.csv',
    'results/t3-medium_100u_12i_stats.csv',
]

consolidate_results(candidate_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48,0.0104,0.4992
t3.small,7370,54,0.007327,True,780,True,True,100,24,0.0208,0.4992
t3.medium,6542,0,0.000000,True,1900,True,True,100,12,0.0416,0.4992


#### Análise de Escalabilidade - Família `m5`

Sigamos para os testes de escalabilidade horizontal de instâncias das famílias `m5`, a saber:

In [22]:
review_instances('m5')

,Instance type,Instance family,Cost,Max Instances
79,m5.large,m5,0.096,5
176,m5.xlarge,m5,0.192,2
285,m5.2xlarge,m5,0.384,1


Os resultados com o máximo de instâncias possível para 500 usuários são apresentados nas tabelas adiante.

In [23]:
m5_results_csvs = [
    'results/m5-large_500u_5i_stats.csv',
    'results/m5-xlarge_500u_2i_stats.csv',
    'results/m5-2xlarge_500u_1i_stats.csv'
]

combine_aggregated_results(m5_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
m5.large,5506,1927,0.349982,False,60000,False,False,500,5
m5.xlarge,8958,4906,0.547667,False,10000,False,False,500,2
m5.2xlarge,8908,5435,0.610126,False,10000,False,False,500,1


Conclusões acerca dos testes com as instâncias da família `m5`:

- Novamente, nenhuma configuração foi capaz de suportar os 500 usuários dentro dos parâmetros exigidos.
- A partir da configuração `m5-xlarge`, a taxa de P95 aproximou-se do valor desejado.
- A confoguração `m5-2xlarge` apresentou as piores taxas de erro dentre os três tipos testados.

Considerando que a família `m5` é de uso geral, devemos considerar a hipótese de que 500 usuários é muita coisa para o nosso orçamento limitado. Neste caso, devemos considerar reduzir o número de usuários, passando a testar com 250 usuários.

Vamos ver como a família `m5` se comporta com 250 usuários.


In [24]:
m5_results_csvs = [
    'results/m5-large_250u_5i_stats.csv',
    'results/m5-xlarge_250u_2i_stats.csv',
    'results/m5-2xlarge_250u_1i_stats.csv'
]

combine_aggregated_results(m5_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
m5.large,3594,246,0.068447,False,60000,False,False,250,5
m5.xlarge,3751,859,0.229006,False,60000,False,False,250,2
m5.2xlarge,5105,1590,0.311459,False,10000,False,False,250,1


Conclusões para esta experiência de 250 usuários com a família `m5`:

- Embora as taxas de erro tenham sido drasticamente reduzidas, a métrica ainda não atinge o objetivo de 1%.
- A latência de P95 permanece bastante alta, inviabilizando o uso da família `m5` com esta configuração de 250 usuários.

Desta forma, vamos, novamente, reduzir o quantitativo de usuários de referência, adotando o mínimo de 100 usuários.

Testando novamente com a família `m5` e 100 usuários:

In [25]:
m5_results_csvs = [
    'results/m5-large_100u_5i_stats.csv',
    'results/m5-xlarge_100u_2i_stats.csv',
    'results/m5-2xlarge_100u_1i_stats.csv'
]

combine_aggregated_results(m5_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
m5.large,2228,79,0.035458,False,51000,False,False,100,5
m5.xlarge,3073,43,0.013993,False,15000,False,False,100,2
m5.2xlarge,3637,31,0.008524,True,2900,True,True,100,1


Observando estes resultados: 

- Finalmente, voltamos a ter uma nova configuração que atende aos requisitos do projeto. É o caso da `m5.2xlarge`, que atinge as métricas de P95 e taxa de erros nos resultados agregados, embora apresente um P95 fora dos critérios para o as chamadas `GET` da raiz do site.

O custo de manter a instância m5.x2large é:

In [26]:
cost_of('m5.2xlarge', 1)

'US$ 0.38'

Com isto, vamos manter o quantitativo de 100 usuários como métrica de referência para nossos experimentos e, até agora, temos as seguintes configurações atendendo os parâmetros:

In [54]:
m5_candidates = ['results/m5-2xlarge_100u_1i_stats.csv']

candidate_csvs = t3_candidates + m5_candidates

consolidate_results(candidate_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48,0.0104,0.4992
t3.small,7370,54,0.007327,True,780,True,True,100,24,0.0208,0.4992
t3.medium,6542,0,0.000000,True,1900,True,True,100,12,0.0416,0.4992
m5.2xlarge,3637,31,0.008524,True,2900,True,True,100,1,0.3840,0.3840


#### Análise de Escalabilidade - Família `c5`

Sigamos para o teste dos tipos de instância da família `c5`, otimizada para computação.

In [29]:
review_instances('c5')

,Instance type,Instance family,Cost,Max Instances
61,c5.large,c5,0.085,5
152,c5.xlarge,c5,0.170,2
259,c5.2xlarge,c5,0.340,1


In [30]:
c5_results_csvs = [
    'results/c5-large_100u_5i_stats.csv',
    'results/c5-xlarge_100u_2i_stats.csv',
    'results/c5-2xlarge_100u_1i_stats.csv'
]

combine_aggregated_results(c5_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
c5.large,3864,0,0.00000,True,4800,True,True,100,5
c5.xlarge,3011,52,0.01727,False,27000,False,False,100,2
c5.2xlarge,4080,4,0.00098,True,1300,True,True,100,1


Observando os resultados obtidos com a família `c5`:

- Por ser otimizada para computação, algo importante para nossa aplicação, um servidor web, a família `c5` apresentou melhores resultados em comparação com a família `m5`.
- De maneira geral, os tipos de instância da família `c5` atingiram as métricas de desempenho exigidas, à exceção, curiosamente, da `c5.xlarge`, o que pode ter sido causado por alguma situação isolada na nuvem. Esse teste merece ser repetido para confirmar os resultados.
- Embora a `c5.large` tenha atingido taxa de erro zero, a `c5.2xlarge` teve latência P95 quase quatro vezes mais baixa que a primeira, mantendo uma taxa de erros de 0,09%.

Devido à divergência inusitada da `c5.xlarge`, vamos repetir os testes de carga com esse tipo de instância para confirmar os resultados.

In [31]:
evaluate_results_from_csv('results/c5-xlarge_100u_2i_proof_stats.csv')

,Type,Name,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass
0,GET,/,948,0,0.0,True,2200,True,True
1,GET,/post-detail,3184,0,0.0,True,2300,True,True
2,NaN,Aggregated,4132,0,0.0,True,2300,True,True


In [32]:
cost_of('c5.large', 5), cost_of('c5.xlarge', 2), cost_of('c5.2xlarge', 1)

('US$ 0.43', 'US$ 0.34', 'US$ 0.34')

Agora sim, temos os resultados da `c5.xlarge` compatíveis com os demais tipos da família. 

Nossas alternativas agora são: 

In [56]:
c5_candidates = [
    'results/c5-large_100u_5i_stats.csv',
    'results/c5-xlarge_100u_2i_proof_stats.csv',
    'results/c5-2xlarge_100u_1i_stats.csv'
]

candidate_csvs = t3_candidates + m5_candidates + c5_candidates

consolidate_results(candidate_csvs)


,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48,0.0104,0.4992
t3.small,7370,54,0.007327,True,780,True,True,100,24,0.0208,0.4992
c5.2xlarge,4080,4,0.000980,True,1300,True,True,100,1,0.3400,0.3400
t3.medium,6542,0,0.000000,True,1900,True,True,100,12,0.0416,0.4992
c5.xlarge,4132,0,0.000000,True,2300,True,True,100,2,0.1700,0.3400
m5.2xlarge,3637,31,0.008524,True,2900,True,True,100,1,0.3840,0.3840
c5.large,3864,0,0.000000,True,4800,True,True,100,5,0.0850,0.4250


#### Análise de Escalabilidade - Família `r5`

Vamos agora fazer os testes com 100 usuários para os tipos de instância da família `r5`, otimizada para memória.

In [34]:
review_instances('r5')

,Instance type,Instance family,Cost,Max Instances
111,r5.large,r5,0.126,3
218,r5.xlarge,r5,0.252,1


In [35]:
r5_results_csvs = [
    'results/r5-large_100u_3i_stats.csv',
    'results/r5-xlarge_100u_1i_stats.csv',
    'results/r5-xlarge_100u_1i_proof_stats.csv'
]

combine_aggregated_results(r5_results_csvs)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count
Instance type,,,,,,,,,
r5.large,2505,0,0.000000,True,5700,True,True,100,3
r5.xlarge,1867,130,0.069630,False,60000,False,False,100,1
r5.xlarge,1841,103,0.055948,False,60000,False,False,100,1


In [36]:
cost_of('r5.large', 3)

'US$ 0.38'

Eis nossas observações em relação aos testes com os dois tipos de instância da família `r5`:

- O tipo `r5.large` atende aos requisitos mínimos de desempenho, com uma taxa de erros de 0 e latência P95 de 5700 ms.
- Todavia, o tipo `r5.xlarge` não conseguiu atender aos parâmetros mínimos de desempenho, o que foi confirmado em uma segunda rodada de testes com esta configuração.

Atualizando nossa planilha de alternativas, temos agora: 

In [57]:
r5_candidates = ['results/r5-large_100u_3i_stats.csv']

candidate_csvs = t3_candidates + m5_candidates + c5_candidates + r5_candidates

df_results = consolidate_results(candidate_csvs)
df_results

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48,0.0104,0.4992
t3.small,7370,54,0.007327,True,780,True,True,100,24,0.0208,0.4992
c5.2xlarge,4080,4,0.000980,True,1300,True,True,100,1,0.3400,0.3400
t3.medium,6542,0,0.000000,True,1900,True,True,100,12,0.0416,0.4992
c5.xlarge,4132,0,0.000000,True,2300,True,True,100,2,0.1700,0.3400
m5.2xlarge,3637,31,0.008524,True,2900,True,True,100,1,0.3840,0.3840
c5.large,3864,0,0.000000,True,4800,True,True,100,5,0.0850,0.4250
r5.large,2505,0,0.000000,True,5700,True,True,100,3,0.1260,0.3780


#### *Ranqueamento* de Tipos de Instância

Vamos agora, classificar nossas alternativas viáveis em termos de custo-benefício.

A partir dos resultados obtidos, podemos concluir que:

- Todas as alternativas viáveis levantadas possuem taxas de erro muito próximas ou iguais a zero.
- A maior variância acontece mesmo no parâmetro de latência P95.
- Apesar de haver variância também em relação ao custo, este não será utilizado como parâmetro de classificação, mantendo-se a prioridade no desempenho e consequente qualidade do serviço.

Revisando a tabela de tipos de instância candidatos:

In [58]:
df_results

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7711,0,0.000000,True,380,True,True,100,48,0.0104,0.4992
t3.small,7370,54,0.007327,True,780,True,True,100,24,0.0208,0.4992
c5.2xlarge,4080,4,0.000980,True,1300,True,True,100,1,0.3400,0.3400
t3.medium,6542,0,0.000000,True,1900,True,True,100,12,0.0416,0.4992
c5.xlarge,4132,0,0.000000,True,2300,True,True,100,2,0.1700,0.3400
m5.2xlarge,3637,31,0.008524,True,2900,True,True,100,1,0.3840,0.3840
c5.large,3864,0,0.000000,True,4800,True,True,100,5,0.0850,0.4250
r5.large,2505,0,0.000000,True,5700,True,True,100,3,0.1260,0.3780


**Os resultados são interessantes.**
Podemos perceber que, embora seja uma família de máquinas mais simples em tese, uma grande quantidade de instâncias `t3.micro` se mostra a opção de melhor desempenho para nosso caso. Investigando a documentação da AWS, acreditamos que isto se explica pelo fato da família `t3` se caracterizar pela chamada [*burstable performance*](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/burstable-performance-instances.html). 

A **burstable performance**, na prática, significa que a instância opera a maior parte do tempo com uma **capacidade de CPU limitada (baseline)**, mas pode **acelerar para 100% da CPU física** sempre que necessário, usando **créditos de CPU acumulados** quando está ociosa. Isso cria um comportamento muito eficiente para workloads que passam longos períodos “tranquilos” e, ocasionalmente, precisam de picos curtos ou moderados de processamento. Enquanto houver créditos, a instância entrega desempenho total como uma máquina “de produção”; quando os créditos acabam, ela volta ao baseline (ou continua acima dele, no modo unlimited, com custo adicional). Em resumo: é como um carro híbrido que anda de forma econômica quase sempre, mas tem potência total disponível quando se pisa fundo.

Adicionalmente, embora não seja critério de classificação, é interessante notar que `t3.micro` também é a alternativa mais barata de todas as opções viáveis.

### Otimização da Aplicação

A linha estratégica de fazer o *tuning* da aplicação precisa levar em conta o tipo de instância alvo, ou seja, as configurações de hardware da instância é que devem orientar as alterações de configuração. Isto é por que a otimização depende estritamente da quantidade de memória e processamento disponíveis.

Neste sentido, considerando os quatro primeiros tipos de instância em nosso *ranking*, temos:

In [66]:
filter = lambda x: x['Instance type'] in ['t3.micro', 't3.small', 'c5.2xlarge', 't3.medium']
columns = ['Instance type', 'vCPUs', 'Memory (GiB)']    

df_prices[df_prices.apply(filter, axis=1)][columns]

,Instance type,vCPUs,Memory (GiB)
2,t3.micro,2,1.0
7,t3.small,2,2.0
18,t3.medium,2,4.0
259,c5.2xlarge,8,16.0


Para esta análise, e em função dos membros da equipe não serem especialistas em Wordpress ou em configuração do Apache HTTP Server, recorreu-se à análise dos cenários de otimização por um modelo de *Large Language Model (LLM)*. Para isto, foi utilizado o modelo do Google **Gemini 3 Pro**, tendo-lhe dado a base de código do projeto, o contexto da otimização e os cenários específicos desejados (registrados na tabela anterior). O plano de otimização, com explicações e recomendações, foi salvo no arquivo [tuning_scenarios.md](docs/tuning_scenarios.md).

Em resumo:

| Instance type | vCPUs | Memória | Cenário | Recomendação (MaxRequestWorkers) |
| :--- | :--- | :--- | :--- | :--- |
| t3.micro | 2 | 1GB | Foco: Evitar OOM (Out of Memory). É o cenário de "sobrevivência". | 25 |
| t3.small | 2 | 2GB | Foco: Escalar proporcionalmente. 2GB permite dobrar a concorrência do cenário anterior. | 50 |
| t3.medium | 2 | 4GB | Foco: O gargalo começa a ser a quantidade de vCPUs. | 100 |
| c5.2xlarge | 8 | 16GB | Foco: Máquina robusta, permite maior utilização. Gargalo provável começa a ser a rede ou banco de dados. | 450 |


#### Cenário de Otimização para t3.micro

Vamos fazer os testes com o cenário otimizado para t3.micro, o que, na verdade, já se encontra muito similar ao cenário original, que adotamos para os testes.

Fizemos as modificações necessárias no arquivo [`user_data_template.sh`](scripts/data_scripts/user_data_template.sh), reproduzidas abaixo:

```bash
<IfModule mpm_prefork_module>
    StartServers             2
    MinSpareServers          2
    MaxSpareServers          5
    MaxRequestWorkers       25
    ServerLimit             25
</IfModule>
```

Agora, vamos executar os testes para verificar se a otimização foi efetiva. Para isto, vamos testar as camadas de demanda de usuários definidas anteriormente, até o limite de 1000 usuários.


In [74]:
tuned_t3micro_results = [
    'results/tuned_t3-micro_100u_48i_stats.csv',
    'results/tuned_t3-micro_250u_48i_stats.csv',
    'results/tuned_t3-micro_500u_48i_stats.csv',
    'results/tuned_t3-micro_1000u_48i_stats.csv'
]

consolidate_results(tuned_t3micro_results)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.micro,7663,0,0.000000,True,440,True,True,100,48,0.0104,0.4992
t3.micro,18503,0,0.000000,True,720,True,True,250,48,0.0104,0.4992
t3.micro,34041,725,0.021298,False,1200,True,False,500,48,0.0104,0.4992
t3.micro,45882,32686,0.712393,False,9700,True,False,1000,48,0.0104,0.4992


Podemos ver que, com o máximo de instâncias possível (48), o tipo `t3.micro` atende, dentro dos parâmetros de performance exigidos, as camadas de 100 e 250 usuários, não sendo mais capaz de responder com taxa de erro aceitável a partir de 500 usuários. Esta configuração é razoável para um síte web de pequena escala.

#### Cenário de otimização para t3.small

Vamos agora, ajustar as configurações da aplicação para otimizá-la para um contexto de 2GB de memória, que é o cenário da instância `t3.small`. A modificação a ser feita no arquivo `user_data_template.sh` é a seguinte:

```bash
<IfModule mpm_prefork_module>
    StartServers             5
    MinSpareServers          5
    MaxSpareServers         10
    MaxRequestWorkers       50
    ServerLimit             50
</IfModule>
```

Vamos ver os resultados:

In [76]:
tuned_t3small_results = [
    'results/tuned_t3-small_100u_24i_stats.csv',
    'results/tuned_t3-small_250u_24i_stats.csv',
    'results/tuned_t3-small_500u_24i_stats.csv',
    'results/tuned_t3-small_1000u_24i_stats.csv'
]

consolidate_results(tuned_t3small_results)

,Request Count,Failure Count,Error Rate,Error Rate Pass,95%,P95 Pass,Total Pass,Users count,Instance count,Cost,Total Cost
Instance type,,,,,,,,,,,
t3.small,7471,0,0.000000,True,640,True,True,100,24,0.0208,0.4992
t3.small,14448,49,0.003391,True,3200,True,True,250,24,0.0208,0.4992
t3.small,37947,32139,0.846944,False,14000,False,False,1000,24,0.0208,0.4992
t3.small,12686,5727,0.451443,False,23000,False,False,500,24,0.0208,0.4992


Com a devida otimização do Apache/PHP, o tipo de instância `t3.small` comporta-se de forma similar à instância `t3.micro`, suportando, dentro dos parâmetros de desempenho estabelecidos, 250 usuários, passando a falhar, neste caso com alta taxa de falha (de mais de 45%) quando se tem 500 usuários.

#### Cenário de Otimização para t3.medium

O ajuste recomendado para teste da t3.medium requer a seguinte configuração no arquivo `httpd.conf`:

```apache
<IfModule mpm_prefork_module>
    StartServers             5
    MinSpareServers          5
    MaxSpareServers         10
    MaxRequestWorkers      100
    ServerLimit            100
</IfModule>
```

Vamos analisar os resultados:

In [ ]:
tuned_t3medium_results = [
    'results/tuned_t3-medium_100u_12i_stats.csv',
    'results/tuned_t3-medium_250u_12i_stats.csv',
    'results/tuned_t3-medium_500u_12i_stats.csv',
    'results/tuned_t3-medium_1000u_12i_stats.csv'
]

consolidate_results(tuned_t3medium_results)